In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import jax
import jax.numpy as jnp
import numpy as np
import dill

In [2]:
batch_data = dill.load(open("debug.pkl", "rb"))
batch = batch_data["batch"]
rewards = batch_data["rewards"]

In [116]:
reset_token_id = 3
gamma = 1.0

@jax.jit
def scan_fn(carry, idx):
    pointer_correct = carry["pointer_correct"]
    actions = carry["actions"]
    target = carry["target"]
    pred_mask = carry["pred_mask"]
    last_reset_idx = carry["last_reset_idx"]
    reset_idxes = carry["reset_idxes"]
    correct_lens = carry["correct_lens"]
    curr_trial = carry["curr_trial"]

    action_match = actions[idx] == target[pointer_correct]
    is_reset = actions[idx] == reset_token_id
    is_reset_with_pred = jnp.logical_and(is_reset, pred_mask[idx])

    # Shift the pointer if the action matches the target and we're within a prediction mask
    reset_pointer = jax.lax.select(
        is_reset,
        1,
        0,
    )
    pointer_correct = jax.lax.select(
        action_match,
        pointer_correct + 1, # Increment pointer by 1 if current token matches
        reset_pointer, # Reset to 0 if it's a mistake and not a reset token, to 1 otherwise
    )

    # jax.debug.print(
    #     "idx: {idx}, pc={pointer_correct}, trial={curr_trial}, cl={correct_lens}",
    #     idx=idx,
    #     pointer_correct=pointer_correct,
    #     curr_trial=curr_trial,
    #     correct_lens=correct_lens,
    # )

    # Update the last reset index to current index upon new trial
    last_reset_idx = jax.lax.select(
        is_reset_with_pred,
        idx,
        last_reset_idx,
    )

    reset_idxes = reset_idxes.at[curr_trial + 1].set(
        jax.lax.select(
            is_reset_with_pred,
            last_reset_idx,
            reset_idxes[curr_trial + 1],
        )
    )

    correct_lens = correct_lens.at[curr_trial].set(
        jnp.maximum(pointer_correct, correct_lens[curr_trial])
    )

    curr_trial = jax.lax.select(
        is_reset_with_pred,
        curr_trial + 1,
        curr_trial,
    )

    return {
        "pointer_correct": pointer_correct,
        "last_reset_idx": last_reset_idx,
        "actions": actions,
        "target": target,
        "pred_mask": pred_mask,
        "reset_idxes": reset_idxes,
        "correct_lens": correct_lens,
        "curr_trial": curr_trial,
    }, None

In [149]:
returns = np.zeros(batch["observations"].shape)

sample_i = 20
# sample_i = 10
# sample_i = 0
(pred_mask, actions, target) = (batch["pred_mask"][sample_i], batch["actions"][sample_i], batch["target"][sample_i])

pointer_correct = np.array(1, dtype=int)
last_reset_idx = np.array(-1, dtype=int)
curr_trial = np.array(0, dtype=int)
reset_idxes = np.full_like(actions, fill_value=-1, dtype=int)
reset_idxes[0] = np.where(pred_mask == 1)[0][0] - 1
correct_lens = np.full_like(actions, fill_value=-1, dtype=int)
last_idx = min(np.where(pred_mask == 1)[0][-1] + 2, actions.shape[-1])

res, _ = jax.lax.scan(
    scan_fn,
    {
        "pointer_correct": pointer_correct,
        "last_reset_idx": last_reset_idx,
        "actions": actions.at[:reset_idxes[0] + 1].set(reset_token_id),
        "target": target,
        "pred_mask": pred_mask.astype(int),
        "reset_idxes": reset_idxes,
        "correct_lens": correct_lens,
        "curr_trial": curr_trial,
    },
    np.arange(last_idx),
)

reset_idxes = res["reset_idxes"]
correct_lens = res["correct_lens"]
correct_lens = np.concatenate(([0], correct_lens))

last_idx = min(np.where(pred_mask == 1)[0][-1] + 1, actions.shape[-1])
# cum_correct_lens = np.maximum.accumulate(correct_lens)
# improvements = correct_lens[1:] - cum_correct_lens[:-1]
improvements = correct_lens[1:] - correct_lens[:-1]
reset_idxes = reset_idxes.at[(np.where(reset_idxes == -1))[0][0]].set(last_idx)
trial_lengths = np.diff(reset_idxes[reset_idxes != -1])

# Update the returns array
returns[sample_i, reset_idxes[0]:last_idx] = np.repeat(
    gamma ** (
        np.arange(int(np.sum(reset_idxes != -1)) - 1)
    ) * improvements[:int(np.sum(reset_idxes != -1)) - 1],
    trial_lengths,
)

In [150]:
print(target)
start_pred_idx = np.where(pred_mask)[0][0].item()
print(start_pred_idx)

[3 0 0 0 0 0 1 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6]
11


In [151]:
reset_idxes

Array([10, 43, 49, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],      dtype=int32)

In [152]:
correct_lens

array([ 0,  6,  7, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])

In [153]:
returns[sample_i, reset_idxes[0]:last_idx]

array([6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6.,
       6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 1.,
       1., 1., 1., 1., 1.])

In [154]:
reset_idxes[0], last_idx

(Array(10, dtype=int32), np.int64(49))

In [155]:
batch["observations"][sample_i][np.where(pred_mask)[0]]

Array([3, 0, 0, 0, 0, 2, 4, 4, 2, 4, 2, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 2,
       4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 3, 0, 0, 0, 0, 0], dtype=int32)

In [156]:
batch["actions"][sample_i][np.where(pred_mask)[0]]

Array([0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2,
       0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0], dtype=int32)